In [ ]:
#Task 1.01: Counting pairs
from collections import Counter

type Pair = tuple[int, int]
def count(ids: list[int]) -> dict[Pair, int]:
    """Count every adjacent pair of token IDs."""
    return dict(Counter(zip(ids, ids[1:])))

# Small test
assert count([1, 2, 1, 2, 3]) == {(1, 2): 2, (2, 1): 1, (2, 3): 1}

In [ ]:
#Task 1.02: Replacing pairs
def replace(ids: list[int], pair: Pair, new_id: int) -> list[int]:
    """Replace non-overlapping occurrences of pair from left to right."""
    result: list[int] = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            result.append(new_id)
            i += 2
        else:
            result.append(ids[i])
            i += 1
    return result

# Small tests
assert replace([1, 2, 1, 2, 3], (1, 2), 9) == [9, 9, 3]
assert replace([1, 1, 1], (1, 1), 9) == [9, 1]

In [ ]:
#Task 1.03: Encoding and decoding
class Tokenizer:
    def __init__(self) -> None:
        self.merges: dict[Pair, int] = {}
        self.vocab: dict[int, bytes] = {i: bytes([i]) for i in range(2**8)}

    def encode(self, text: str) -> list[int]:
        ids = list(text.encode("utf-8"))
        while True:
            counts = count(ids)
            mergeable_pairs = counts.keys() & self.merges.keys()
            if not mergeable_pairs:
                break
            # New token IDs are assigned in training order. Apply the earliest
            # applicable learned merge, i.e. the one with the lowest ID.
            to_merge = min(mergeable_pairs, key=self.merges.__getitem__)
            ids = replace(ids, to_merge, self.merges[to_merge])
        return ids

    def decode(self, ids: list[int]) -> str:
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")

In [ ]:
#Task 1.04: Training a tokeniser
def from_text(text: str, vocab_size: int) -> Tokenizer:
    """Induce a byte-level BPE tokenizer from text."""
    if vocab_size < 256:
        raise ValueError("vocab_size must be at least 256")

    tok = Tokenizer()
    ids = list(text.encode("utf-8"))

    for new_id in range(256, vocab_size):
        pair_counts = count(ids)
        if not pair_counts:
            break

        # Python preserves insertion order. For a frequency tie, this selects
        # the pair first encountered in the left-to-right scan in count().
        best_pair = max(pair_counts, key=pair_counts.__getitem__)
        ids = replace(ids, best_pair, new_id)
        tok.merges[best_pair] = new_id
        tok.vocab[new_id] = tok.vocab[best_pair[0]] + tok.vocab[best_pair[1]]

    return tok

# Small round-trip test
test_tok = from_text("banana banana", 270)
assert test_tok.decode(test_tok.encode("banana")) == "banana"

def save(tokenizer: Tokenizer, filename: str) -> None:
    with open(filename, "w", encoding="utf-8") as f:
        for fst, snd in tokenizer.merges:
            print(f"{fst} {snd}", file=f)

#Task 1.05: Tokenisation quirks

creativecommons → evitaercemmo
MERCHANTABILITY → TIBILAHCrem
NSNotification → noitacifitoNS
authentication → noitacitnehtua

How many of these words come out right? 
Only one authenication

What happens when you modify the prompt and explicitly disable “thinking” and external tools?
when you prompt explicitly to disable thinking and external tools, gives correct answers
creativecommons → snommocevitaerc
MERCHANTABILITY → YTILIBATNAHCREM
NSNotification → noitacifitoNSN
authentication → noitacitnehtua


What could be the problem when words come out wrong? Generate ideas by inspecting the words in Tiktokenizer. Try to come up with other prompts that illustrate problems related to tokenisation.

Tokenization may contribute because the model processes text as tokens (often groups of letters), not necessarily one character at a time.But the tokenizer itself isn't changing the word. The model is making the mistake when generating the reversed answer.



In [ ]:
#Task 1.06: Tokenisation and multi-linguality
# Requires wiki-en-1m.txt and wiki-is-1m.txt in the notebook directory.
# Training on one million characters may take several minutes.
with open("wiki-en-1m.txt", encoding="utf-8") as f:
    en_text = f.read()
with open("wiki-is-1m.txt", encoding="utf-8") as f:
    is_text = f.read()

# Use the vocabulary size required by your course instructions or provided tokenizer.
en_tok = from_text(en_text, vocab_size=1024)
en_ids = en_tok.encode(en_text)
is_ids = en_tok.encode(is_text)

def context_statistics(text: str, ids: list[int], context_length: int = 1024) -> dict[str, float]:
    chars_per_token = len(text) / len(ids)
    return {
        "characters": len(text),
        "tokens": len(ids),
        "characters_per_token": chars_per_token,
        "estimated_characters_in_1024_tokens": context_length * chars_per_token,
    }

print("English:", context_statistics(en_text, en_ids))
print("Icelandic with English tokenizer:", context_statistics(is_text, is_ids))

In [ ]:
#Part 2: Embeddings
#Task 1.07: Bag-of-words classifier

class ReviewDataset(Dataset):
    def __init__(self, filename: str, label: int = 0) -> None:
        with open(filename, encoding="utf-8") as f:
            tokenized_lines = [line.split() for line in f]
        self.items: list[Item] = [(tokens[2:], tokens[label]) for tokens in tokenized_lines]

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> Item:
        return self.items[idx]


type Item = tuple[list[str], str]

In [ ]:
#Task 1.08: Vectoriser
class ReviewVectorizer:
    PAD = "[PAD]"
    UNK = "[UNK]"

    def __init__(self, dataset: ReviewDataset, n_vocab: int = 1024) -> None:
        # zip(*dataset) transposes review-label pairs into two tuples.
        reviews, labels = zip(*dataset)

        counter = Counter(token for review in reviews for token in review)
        most_common = [token for token, _ in counter.most_common(n_vocab - 2)]

        # Reserve ID 0 for padding and ID 1 for unknown tokens.
        self.t2i = {token: i for i, token in enumerate([self.PAD, self.UNK] + most_common)}
        self.l2i = {label: i for i, label in enumerate(sorted(set(labels)))}

    def __call__(self, items: list[Item]) -> tuple[torch.Tensor, torch.Tensor]:
        reviews, labels = zip(*items)
        pad_id = self.t2i[self.PAD]
        unk_id = self.t2i[self.UNK]
        max_length = max(len(review) for review in reviews)

        x = [
            [self.t2i.get(token, unk_id) for token in review]
            + [pad_id] * (max_length - len(review))
            for review in reviews
        ]
        y = [self.l2i[label] for label in labels]

        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [ ]:
#Task 1.09: Configurable training loop
def train(
    filename: str = "reviews-train.txt",
    label: int = 0,
    n_vocab: int = 1024,
    embedding_dim: int = 64,
    learning_rate: float = 0.001,
    batch_size: int = 16,
    epochs: int = 10,
    shuffle: bool = True,
    kaiming_embedding: bool = False,
) -> tuple[ReviewVectorizer, Classifier]:
    # Load labelled reviews and learn vocabulary/label mappings from them.
    dataset = ReviewDataset(filename, label=label)
    vectorizer = ReviewVectorizer(dataset, n_vocab=n_vocab)

    # Build the embedding-plus-linear classifier.
    model = Classifier(n_vocab, embedding_dim, len(vectorizer.l2i))
    if kaiming_embedding:
        # This matches nn.Linear.reset_parameters() for the weight matrix.
        nn.init.kaiming_uniform_(model.embedding.weight, a=math.sqrt(5))

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    data_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=vectorizer,
    )

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for bx, by in data_loader:
            optimizer.zero_grad()              # Clear gradients from prior batch.
            logits = model(bx)                 # Forward pass to obtain class logits.
            loss = F.cross_entropy(logits, by) # Compare logits with gold labels.
            loss.backward()                    # Backpropagation computes gradients.
            optimizer.step()                   # Adam updates all trainable parameters.
            running_loss += loss.item()

        print(f"Epoch {epoch + 1}, loss: {running_loss / len(data_loader):.4f}")

    return vectorizer, model


In [ ]:
#Task 1.10: Train both prediction tasks
# label=0 predicts product category: camera versus music.
torch.manual_seed(42)
category_vectorizer, category_model = train(label=0)

# label=1 predicts sentiment: neg versus pos.
torch.manual_seed(42)
sentiment_vectorizer, sentiment_model = train(label=1)

# Compare the printed loss curves. The higher/slower loss task is harder
# for this model and this dataset. The fixed seed makes random initialization
# and shuffled batch order reproducible across the two runs.

In [ ]:
#Task 1.11: Export and inspect embeddings
def save_embeddings(
    vectorizer: ReviewVectorizer,
    model: Classifier,
    vectors_filename: str,
    metadata_filename: str,
) -> None:
    i2t = {i: token for token, i in vectorizer.t2i.items()}
    embeddings = model.embedding.weight.detach().cpu().numpy()
    with open(vectors_filename, "wt", encoding="utf-8") as f1, open(
        metadata_filename, "wt", encoding="utf-8"
    ) as f2:
        for i, embedding in enumerate(embeddings):
            print("\t".join(f"{value:.5f}" for value in embedding), file=f1)
            print(i2t[i], file=f2)

save_embeddings(
    category_vectorizer, category_model,
    "category-vectors.tsv", "category-metadata.tsv",
)
save_embeddings(
    sentiment_vectorizer, sentiment_model,
    "sentiment-vectors.tsv", "sentiment-metadata.tsv",
)


In [ ]:
#Task 1.12: Kaiming initialisation
# nn.Linear uses kaiming_uniform_ with a=sqrt(5) for its weight matrix.
# Compare default embedding initialization and the matching Kaiming setup.
torch.manual_seed(42)
default_vectorizer, default_model = train(label=0, kaiming_embedding=False)

torch.manual_seed(42)
kaiming_vectorizer, kaiming_model = train(label=0, kaiming_embedding=True)

save_embeddings(
    default_vectorizer, default_model,
    "default-vectors.tsv", "default-metadata.tsv",
)
save_embeddings(
    kaiming_vectorizer, kaiming_model,
    "kaiming-vectors.tsv", "kaiming-metadata.tsv",
)